# 第十讲：SciPy 统计分析

**学习目标**
- 理解假设检验的核心逻辑：P 值到底是什么？
- 掌握正态性检验：Jarque-Bera 检验 & Kolmogorov-Smirnov 检验
- 掌握 t 检验：单样本 / 双样本 / 配对
- 掌握 Pearson / Spearman 相关系数及其显著性检验
- 掌握线性回归（OLS）的基本实现与解读
- 实战：检验上证指数收益率是否服从正态分布，分析行业间相关性

> 💡 统计检验是量化分析的「裁判」—— 你的策略收益是真本事还是运气？因子之间有没有真正的关系？这些都需要统计检验来回答。

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print(f"SciPy 版本: {stats.__version__ if hasattr(stats, '__version__') else '内嵌'}")
print("所有库加载成功！")

## 10.1 假设检验与 P 值 —— 统计学的「法庭逻辑」

假设检验的思路和法庭审判一模一样：

| 法庭 | 统计检验 |
|------|----------|
| 无罪推定 | 原假设 H₀（默认「没有效果」） |
| 检察官举证 | 计算检验统计量 |
| 证据强度 | P 值 |
| 判有罪（证据充分） | 拒绝 H₀（P < α） |
| 判无罪（证据不足） | 不拒绝 H₀（P ≥ α） |

**P 值的严格定义**：假设 H₀ 为真，观测到当前结果（或更极端结果）的概率。

- P 值越小 → 数据与 H₀ 矛盾越大 → 越有理由拒绝 H₀
- 常见显著性水平 α = 0.05（5%）或 α = 0.01（1%）

**⚠️ 常见误解**：
- ❌ P 值不是「H₀ 为真的概率」
- ❌ P = 0.03 不代表「有 97% 的把握」
- ❌ 不拒绝 H₀ ≠ H₀ 为真（只是证据不够）

> 理解 P 值，是理解所有后续检验的基础。

In [ ]:
# 用抛硬币实验直觉理解 P 值
# 假设你怀疑一枚硬币不均匀，抛了 100 次，出现 60 次正面
# H₀: 硬币是均匀的（p=0.5）
# H₁: 硬币不均匀（p≠0.5）

from scipy.stats import binom

n_flips = 100
n_heads = 60   # 观测到 60 次正面
p0 = 0.5       # H₀ 假设的概率

# 双尾检验：计算 P(X >= 60) + P(X <= 40)
# 因为 60 偏离 50 的距离是 10，所以也要算另一端 40
p_value_upper = 1 - binom.cdf(n_heads - 1, n_flips, p0)  # P(X >= 60)
p_value_lower = binom.cdf(n_flips - n_heads, n_flips, p0)  # P(X <= 40)
p_value = p_value_upper + p_value_lower

print(f"抛 {n_flips} 次，出现 {n_heads} 次正面")
print(f"双尾 P 值 = {p_value:.4f}")
print(f"在 α=0.05 水平下: {'拒绝 H₀ → 硬币可能不均匀' if p_value < 0.05 else '不拒绝 H₀ → 证据不足'}")

# 可视化：在 H₀ 下，100 次抛掷正面次数的分布
x = np.arange(35, 66)
y = binom.pmf(x, n_flips, p0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, y, color='steelblue', alpha=0.7, label='H₀ 下的理论分布')
ax.axvline(n_heads, color='red', linestyle='--', linewidth=2, label=f'观测值 = {n_heads}')
# 标注拒绝域
ax.fill_between(x[x <= 40], binom.pmf(x[x <= 40], n_flips, p0), color='red', alpha=0.3, label='拒绝域')
ax.fill_between(x[x >= 60], binom.pmf(x[x >= 60], n_flips, p0), color='red', alpha=0.3)
ax.set_xlabel('正面次数')
ax.set_ylabel('概率')
ax.set_title(f'抛硬币实验：P 值 = {p_value:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

## 10.2 正态性检验 —— 你的数据是「正态」的吗？

很多统计方法（t 检验、线性回归、VaR 参数法）都假设数据服从正态分布。如果这个假设不成立，结论可能不可靠。

### 两种常用检验

| 检验方法 | 全称 | 原假设 H₀ | 适用场景 |
|----------|------|-----------|----------|
| **JB 检验** | Jarque-Bera | 数据服从正态分布 | 检查偏度和峰度是否符合正态 |
| **KS 检验** | Kolmogorov-Smirnov | 数据服从指定分布 | 比较经验分布与理论分布的整体差异 |

**关键区别**：
- JB 检验只看偏度（是否对称）和峰度（是否太尖/太平）
- KS 检验直接比较整个分布形状，更全面

**解读规则**：
- P < α → 拒绝 H₀ → 数据**不服从**正态分布
- P ≥ α → 不拒绝 H₀ → 没有足够证据说数据**不**服从正态

In [ ]:
# 先用「已知答案」的数据建立直觉

np.random.seed(42)

# 生成两组数据
normal_data = np.random.normal(loc=0, scale=1, size=1000)    # 真正的正态分布
uniform_data = np.random.uniform(low=-3, high=3, size=1000)  # 均匀分布（不是正态）

print("=" * 60)
print("正态数据 的统计量:")
print(f"  均值={normal_data.mean():.4f}, 标准差={normal_data.std():.4f}")
print(f"  偏度={stats.skew(normal_data):.4f}, 峰度={stats.kurtosis(normal_data):.4f}")

print(f"\n均匀数据 的统计量:")
print(f"  均值={uniform_data.mean():.4f}, 标准差={uniform_data.std():.4f}")
print(f"  偏度={stats.skew(uniform_data):.4f}, 峰度={stats.kurtosis(uniform_data):.4f}")

# JB 检验
print("\n" + "=" * 60)
print("Jarque-Bera 正态性检验")
print("-" * 60)

jb_stat_normal, jb_p_normal = stats.jarque_bera(normal_data)
jb_stat_uniform, jb_p_uniform = stats.jarque_bera(uniform_data)

print(f"正态数据: JB统计量={jb_stat_normal:.4f}, P值={jb_p_normal:.4f} → {'正态' if jb_p_normal > 0.05 else '非正态'}")
print(f"均匀数据: JB统计量={jb_stat_uniform:.4f}, P值={jb_p_uniform:.6f} → {'正态' if jb_p_uniform > 0.05 else '非正态'}")

# KS 检验
print("\n" + "=" * 60)
print("Kolmogorov-Smirnov 正态性检验")
print("-" * 60)

ks_stat_normal, ks_p_normal = stats.kstest(normal_data, 'norm', args=(normal_data.mean(), normal_data.std()))
ks_stat_uniform, ks_p_uniform = stats.kstest(uniform_data, 'norm', args=(uniform_data.mean(), uniform_data.std()))

print(f"正态数据: KS统计量={ks_stat_normal:.4f}, P值={ks_p_normal:.4f} → {'正态' if ks_p_normal > 0.05 else '非正态'}")
print(f"均匀数据: KS统计量={ks_stat_uniform:.4f}, P值={ks_p_uniform:.6f} → {'正态' if ks_p_uniform > 0.05 else '非正态'}")

In [ ]:
# 可视化：正态 vs 均匀 的分布对比

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title, jb_p in zip(
    axes, 
    [normal_data, uniform_data], 
    ['正态分布数据', '均匀分布数据'],
    [jb_p_normal, jb_p_uniform]
):
    ax.hist(data, bins=40, density=True, alpha=0.7, color='steelblue', edgecolor='white')
    
    # 叠加理论正态曲线
    x = np.linspace(data.min(), data.max(), 200)
    ax.plot(x, stats.norm.pdf(x, data.mean(), data.std()), 'r-', linewidth=2, label='理论正态曲线')
    
    ax.set_title(f'{title}\nJB P值 = {jb_p:.4f} → {"正态 ✓" if jb_p > 0.05 else "非正态 ✗"}')
    ax.set_xlabel('值')
    ax.set_ylabel('密度')
    ax.legend()

plt.tight_layout()
plt.show()

## 10.3 t 检验 —— 比较均值是否有显著差异

t 检验是量化分析中最常用的检验之一，用于回答：
- 策略的平均收益是否显著不为零？（单样本 t 检验）
- 两个策略的收益是否有显著差异？（双样本 t 检验）
- 改进前后的策略是否有显著变化？（配对 t 检验）

### 三种 t 检验

| 类型 | H₀ | 适用场景 |
|------|-----|----------|
| 单样本 | μ = μ₀ | 检验均值是否等于某个值 |
| 双样本独立 | μ₁ = μ₂ | 比较两组独立样本的均值 |
| 配对 | μ_diff = 0 | 同一样本处理前后的差异 |

**前提假设**：
- 数据近似正态分布（或样本量足够大，中心极限定理保护）
- 双样本 t 检验还要求两组方差相等（可用 Levene 检验验证）

In [ ]:
# === 单样本 t 检验 ===
# 场景：检验某策略的日均收益率是否显著不为零

np.random.seed(42)
strategy_returns = np.random.normal(loc=0.0005, scale=0.015, size=252)  # 一年交易日

print("单样本 t 检验: 策略日均收益率是否 ≠ 0")
print("H₀: μ = 0（策略没有超额收益）")
print("H₁: μ ≠ 0（策略有超额收益）")
print("-" * 50)

t_stat, t_pvalue = stats.ttest_1samp(strategy_returns, 0)

print(f"样本均值: {strategy_returns.mean():.6f}")
print(f"样本标准差: {strategy_returns.std():.6f}")
print(f"t 统计量: {t_stat:.4f}")
print(f"P 值: {t_pvalue:.4f}")
print(f"结论: {'拒绝 H₀ → 策略收益显著不为零' if t_pvalue < 0.05 else '不拒绝 H₀ → 收益不显著'}")

# 年化收益和夏普比
annual_return = strategy_returns.mean() * 252
annual_vol = strategy_returns.std() * np.sqrt(252)
sharpe = annual_return / annual_vol
print(f"\n年化收益: {annual_return:.2%}, 年化波动: {annual_vol:.2%}, 夏普比: {sharpe:.2f}")

In [ ]:
# === 双样本 t 检验 ===
# 场景：比较两个策略的收益是否有显著差异

np.random.seed(42)
strategy_a = np.random.normal(loc=0.0008, scale=0.015, size=252)  # 策略 A
strategy_b = np.random.normal(loc=0.0003, scale=0.018, size=252)  # 策略 B

print("双样本 t 检验: 策略 A 和 B 的收益是否有显著差异")
print("H₀: μ_A = μ_B")
print("-" * 50)

# 先检验方差是否相等（Levene 检验）
lev_stat, lev_p = stats.levene(strategy_a, strategy_b)
print(f"Levene 方差齐性检验: 统计量={lev_stat:.4f}, P值={lev_p:.4f}")
print(f"  → 方差{'相等' if lev_p > 0.05 else '不相等'}，选择{'标准' if lev_p > 0.05 else 'Welch'}t检验")

# Welch t 检验（不假设方差相等，更安全）
t_stat2, t_pvalue2 = stats.ttest_ind(strategy_a, strategy_b, equal_var=False)

print(f"\n策略 A 均值: {strategy_a.mean():.6f}")
print(f"策略 B 均值: {strategy_b.mean():.6f}")
print(f"t 统计量: {t_stat2:.4f}")
print(f"P 值: {t_pvalue2:.4f}")
print(f"结论: {'拒绝 H₀ → 两策略收益有显著差异' if t_pvalue2 < 0.05 else '不拒绝 H₀ → 差异不显著'}")

In [ ]:
# === 配对 t 检验 ===
# 场景：同一个策略优化前后的表现对比（同时间段，控制市场因素）

np.random.seed(42)
before = np.random.normal(loc=0.0003, scale=0.015, size=60)   # 优化前 60 天
after = before + np.random.normal(loc=0.0002, scale=0.005, size=60)  # 优化后，略有提升

print("配对 t 检验: 策略优化前后是否有显著改善")
print("H₀: μ_after - μ_before = 0（优化没有效果）")
print("-" * 50)

t_stat3, t_pvalue3 = stats.ttest_rel(after, before)

diff = after - before
print(f"平均差异: {diff.mean():.6f}")
print(f"差异标准差: {diff.std():.6f}")
print(f"t 统计量: {t_stat3:.4f}")
print(f"P 值: {t_pvalue3:.4f}")
print(f"结论: {'拒绝 H₀ → 优化有显著效果' if t_pvalue3 < 0.05 else '不拒绝 H₀ → 优化效果不显著'}")

## 10.4 相关系数与显著性检验 —— 两个变量真的有关系吗？

相关系数衡量两个变量的线性（或单调）关系强度。但**相关不等于因果**！

| 类型 | 衡量什么 | 取值范围 | 适用场景 |
|------|----------|----------|----------|
| **Pearson** | 线性关系 | [-1, 1] | 连续数据、近似正态 |
| **Spearman** | 单调关系（排名相关） | [-1, 1] | 非正态数据、有离群值 |
| **Kendall** | 排序一致性 | [-1, 1] | 小样本、有序数据 |

**显著性检验**：
- H₀: ρ = 0（两个变量没有相关性）
- H₁: ρ ≠ 0（存在相关性）
- P < α → 相关性是「真的」，不是偶然

In [ ]:
# 用模拟数据理解相关系数

np.random.seed(42)
n = 200

# 三种相关性
x = np.random.normal(0, 1, n)
y_strong = 0.9 * x + np.random.normal(0, 0.3, n)   # 强正相关
y_weak = 0.2 * x + np.random.normal(0, 1, n)        # 弱正相关
y_none = np.random.normal(0, 1, n)                   # 无相关

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, y, label in zip(axes, [y_strong, y_weak, y_none], ['强正相关', '弱正相关', '无相关']):
    # Pearson 相关系数及检验
    r, p = stats.pearsonr(x, y)
    
    ax.scatter(x, y, alpha=0.5, s=15, color='steelblue')
    ax.set_title(f'{label}\nPearson r = {r:.3f}, P = {p:.4f}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    
    # 画回归线
    z = np.polyfit(x, y, 1)
    ax.plot(sorted(x), np.polyval(z, sorted(x)), 'r-', linewidth=2)

plt.tight_layout()
plt.show()

In [ ]:
# Pearson vs Spearman 对比
# 当数据有离群值时，Spearman 更稳健

np.random.seed(42)
n = 100
x = np.random.normal(50, 10, n)
y = 2 * x + np.random.normal(0, 15, n)

# 添加离群值
x_outlier = np.append(x, [10, 90, 95])
y_outlier = np.append(y, [200, -50, 300])

print("Pearson vs Spearman 对比（含离群值数据）")
print("=" * 55)

# 无离群值
r_pearson, p_pearson = stats.pearsonr(x, y)
r_spearman, p_spearman = stats.spearmanr(x, y)
print(f"无离群值: Pearson r={r_pearson:.4f} (P={p_pearson:.4f}), Spearman ρ={r_spearman:.4f} (P={p_spearman:.4f})")

# 有离群值
r_pearson_out, p_pearson_out = stats.pearsonr(x_outlier, y_outlier)
r_spearman_out, p_spearman_out = stats.spearmanr(x_outlier, y_outlier)
print(f"有离群值: Pearson r={r_pearson_out:.4f} (P={p_pearson_out:.4f}), Spearman ρ={r_spearman_out:.4f} (P={p_spearman_out:.4f})")

print(f"\n结论: Pearson 对离群值敏感（{r_pearson:.3f} → {r_pearson_out:.3f}），Spearman 更稳定（{r_spearman:.3f} → {r_spearman_out:.3f}）")

## 10.5 线性回归（OLS）—— 量化因子分析的基础

线性回归是量化金融的核心工具。CAPM、Fama-French 三因子/五因子模型，本质上都是线性回归。

**模型**：$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \epsilon$

**关键输出**：
| 指标 | 含义 |
|------|------|
| 系数 (coef) | 每个自变量对因变量的影响大小 |
| t 值 | 系数是否显著不为零 |
| P 值 | 系数的显著性 |
| R² | 模型解释了多少变异（0~1） |
| 调整 R² | 考虑变量个数后的 R²（更公平） |
| F 统计量 | 整个模型是否显著 |

In [ ]:
# OLS 回归实战：用市场收益率解释个股收益率（简化版 CAPM）

np.random.seed(42)
n = 252  # 一年交易日

# 模拟市场收益率
market_returns = np.random.normal(0.0005, 0.012, n)

# 模拟个股收益率 = alpha + beta * market + epsilon
true_alpha = 0.0002   # 真实 alpha
true_beta = 1.3       # 真实 beta
epsilon = np.random.normal(0, 0.008, n)
stock_returns = true_alpha + true_beta * market_returns + epsilon

# 用 statsmodels 做 OLS 回归
X = sm.add_constant(market_returns)  # 添加截距项
model = sm.OLS(stock_returns, X).fit()

print(model.summary())

In [ ]:
# 提取关键指标并解读

print("OLS 回归结果解读")
print("=" * 55)
print(f"Alpha (截距) = {model.params[0]:.6f}  (真实值: {true_alpha})")
print(f"  → t值 = {model.tvalues[0]:.4f}, P值 = {model.pvalues[0]:.4f}")
print(f"  → {'显著' if model.pvalues[0] < 0.05 else '不显著'} (α=0.05)\n")

print(f"Beta (市场因子) = {model.params[1]:.4f}  (真实值: {true_beta})")
print(f"  → t值 = {model.tvalues[1]:.4f}, P值 = {model.pvalues[1]:.6f}")
print(f"  → {'显著' if model.pvalues[1] < 0.05 else '不显著'} (α=0.05)\n")

print(f"R² = {model.rsquared:.4f} → 模型解释了 {model.rsquared:.1%} 的收益变异")
print(f"调整 R² = {model.rsquared_adj:.4f}")
print(f"F 统计量 = {model.fvalue:.4f}, P值 = {model.f_pvalue:.6f}")
print(f"  → 模型整体{'显著' if model.f_pvalue < 0.05 else '不显著'}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 散点图 + 回归线
axes[0].scatter(market_returns, stock_returns, alpha=0.4, s=10, color='steelblue')
x_line = np.linspace(market_returns.min(), market_returns.max(), 100)
axes[0].plot(x_line, model.params[0] + model.params[1] * x_line, 'r-', linewidth=2,
             label=f'y = {model.params[0]:.5f} + {model.params[1]:.3f}x')
axes[0].set_xlabel('市场收益率')
axes[0].set_ylabel('个股收益率')
axes[0].set_title(f'CAPM 回归: R² = {model.rsquared:.3f}')
axes[0].legend()

# 残差分布
axes[1].hist(model.resid, bins=40, density=True, alpha=0.7, color='steelblue', edgecolor='white')
axes[1].set_xlabel('残差')
axes[1].set_ylabel('密度')
axes[1].set_title('残差分布（应近似正态）')

plt.tight_layout()
plt.show()

## 10.6 实战一：检验上证指数收益率的正态性

金融学的一个经典发现：股票收益率**不服从**正态分布，而是呈现「尖峰厚尾」特征——极端事件发生的概率比正态分布预测的要高。

这对风险管理至关重要：如果你用正态分布假设来计算 VaR，会**低估**极端亏损的风险。

In [ ]:
import akshare as ak

# 获取上证指数数据（近 3 年）
print("正在下载上证指数数据...")
sh_index = ak.stock_zh_index_daily(symbol="sh000001")
sh_index['date'] = pd.to_datetime(sh_index['date'])
sh_index = sh_index.set_index('date').sort_index()

# 取最近 3 年
sh_index = sh_index.last('3Y')

# 计算日收益率
sh_index['return'] = sh_index['close'].pct_change()
returns = sh_index['return'].dropna()

if returns.empty:
    print("⚠️ 上证指数数据获取失败或为空，使用模拟数据演示")
    np.random.seed(42)
    returns = pd.Series(
        np.random.standard_t(df=5, size=750) * 0.012 + 0.0002,
        index=pd.date_range('2023-01-01', periods=750, freq='B'),
        name='return'
    )

print(f"数据区间: {returns.index[0].date()} ~ {returns.index[-1].date()}")
print(f"样本量: {len(returns)} 个交易日")
print(f"\n收益率统计量:")
print(f"  均值:   {returns.mean():.6f}")
print(f"  标准差: {returns.std():.6f}")
print(f"  偏度:   {stats.skew(returns):.4f}  (正态=0)")
print(f"  峰度:   {stats.kurtosis(returns):.4f}  (正态=0, >0 表示厚尾)")

In [ ]:
# 正态性检验

print("上证指数日收益率 正态性检验")
print("=" * 55)

# JB 检验
jb_stat, jb_p = stats.jarque_bera(returns)
print(f"\nJarque-Bera 检验:")
print(f"  统计量 = {jb_stat:.4f}")
print(f"  P 值   = {jb_p:.2e}  (科学记数法)")
print(f"  结论: {'拒绝正态假设 → 收益率不服从正态分布' if jb_p < 0.05 else '不拒绝正态假设'}")

# KS 检验
ks_stat, ks_p = stats.kstest(returns, 'norm', args=(returns.mean(), returns.std()))
print(f"\nKolmogorov-Smirnov 检验:")
print(f"  统计量 = {ks_stat:.4f}")
print(f"  P 值   = {ks_p:.4f}")
print(f"  结论: {'拒绝正态假设 → 收益率不服从正态分布' if ks_p < 0.05 else '不拒绝正态假设'}")

# Shapiro-Wilk 检验（样本量 < 5000 时最有力）
if len(returns) <= 5000:
    sw_stat, sw_p = stats.shapiro(returns)
    print(f"\nShapiro-Wilk 检验:")
    print(f"  统计量 = {sw_stat:.4f}")
    print(f"  P 值   = {sw_p:.2e}")
    print(f"  结论: {'拒绝正态假设' if sw_p < 0.05 else '不拒绝正态假设'}")

In [ ]:
# 可视化：实际分布 vs 正态分布

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 直方图 + 正态曲线
axes[0].hist(returns, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='white', label='实际分布')
x = np.linspace(returns.min(), returns.max(), 200)
axes[0].plot(x, stats.norm.pdf(x, returns.mean(), returns.std()), 'r-', linewidth=2, label='理论正态')
axes[0].set_xlabel('日收益率')
axes[0].set_ylabel('密度')
axes[0].set_title(f'上证指数日收益率分布\nJB P值 = {jb_p:.2e} → 非正态')
axes[0].legend()

# Q-Q 图（更直观地看是否偏离正态）
stats.probplot(returns, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q 图（偏离直线 = 偏离正态）')
axes[1].get_lines()[0].set_markersize(3)
axes[1].get_lines()[0].set_alpha(0.5)

plt.tight_layout()
plt.show()

# 极端事件统计
print("\n极端事件统计:")
print(f"  收益率 > 2σ 的天数: {(returns.abs() > 2 * returns.std()).sum()} 天")
print(f"  收益率 > 3σ 的天数: {(returns.abs() > 3 * returns.std()).sum()} 天")
print(f"  正态分布下 > 3σ 的期望概率: {2 * (1 - stats.norm.cdf(3)):.4%}")
print(f"  实际 > 3σ 的概率: {(returns.abs() > 3 * returns.std()).mean():.4%}")

## 10.7 实战二：行业间收益率相关性分析

分析不同行业指数的收益率相关性，回答：
- 哪些行业高度相关（同涨同跌）？
- 哪些行业相关性低（分散化投资的好选择）？
- 相关性是否显著？

In [ ]:
# 获取申万一级行业指数数据
print("正在下载行业指数数据（申万一级）...")

# 申万行业指数代码（选取代表性行业）
industry_codes = {
    '801010': '农林牧渔',
    '801020': '采掘',
    '801030': '化工',
    '801050': '有色金属',
    '801080': '电子',
    '801110': '家用电器',
    '801120': '食品饮料',
    '801130': '纺织服装',
    '801140': '轻工制造',
    '801150': '医药生物',
    '801160': '公用事业',
    '801170': '交通运输',
    '801180': '房地产',
    '801200': '商业贸易',
    '801210': '休闲服务',
    '801230': '综合',
    '801710': '建筑材料',
    '801720': '建筑装饰',
    '801730': '电气设备',
    '801740': '国防军工',
    '801750': '计算机',
    '801760': '传媒',
    '801770': '通信',
    '801780': '银行',
    '801790': '非银金融',
    '801880': '汽车',
    '801890': '机械设备',
}

# 下载各行业数据
industry_returns = pd.DataFrame()
failed = []

for code, name in industry_codes.items():
    try:
        df = ak.index_hist_sw(symbol=code, period='day')
        df['date'] = pd.to_datetime(df['日期'])
        df = df.set_index('date').sort_index()
        df = df.last('2Y')
        s = df['收盘'].astype(float).pct_change()
        if not s.dropna().empty:
            industry_returns[name] = s
        else:
            failed.append(f"{name}({code}): 数据为空")
    except Exception as e:
        failed.append(f"{name}({code}): {e}")

industry_returns = industry_returns.dropna()

if industry_returns.empty:
    print("⚠️ 行业数据获取失败，使用模拟数据演示")
    np.random.seed(42)
    n_days = 500
    dates = pd.date_range('2024-01-01', periods=n_days, freq='B')
    sim_names = ['银行', '食品饮料', '电子', '医药生物', '房地产', '计算机', '新能源', '军工']
    data = {}
    for name in sim_names:
        data[name] = np.random.randn(n_days) * 0.015 + 0.0002
    industry_returns = pd.DataFrame(data, index=dates)

print(f"\n成功获取 {len(industry_returns.columns)} 个行业数据")
print(f"数据区间: {industry_returns.index[0].date()} ~ {industry_returns.index[-1].date()}")
print(f"样本量: {len(industry_returns)} 个交易日")
if failed:
    print(f"\n获取失败的行业 ({len(failed)} 个):")
    for f in failed[:5]:
        print(f"  {f}")
    if len(failed) > 5:
        print(f"  ... 等共 {len(failed)} 个")

In [ ]:
# 计算相关系数矩阵

corr_matrix = industry_returns.corr(method='pearson')

# 热力图
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

# 设置刻度
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr_matrix.columns, fontsize=9)

# 标注数值
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        val = corr_matrix.iloc[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=ax, label='Pearson 相关系数')
ax.set_title('行业间日收益率相关性矩阵', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 相关系数的显著性检验
# 对每对行业，计算相关系数和 P 值

n_industries = len(industry_returns.columns)
results = []

for i in range(n_industries):
    for j in range(i + 1, n_industries):
        name_i = industry_returns.columns[i]
        name_j = industry_returns.columns[j]
        r, p = stats.pearsonr(industry_returns.iloc[:, i], industry_returns.iloc[:, j])
        results.append({
            '行业A': name_i,
            '行业B': name_j,
            '相关系数': r,
            'P值': p,
            '显著': '✓' if p < 0.05 else '✗'
        })

results_df = pd.DataFrame(results)

# 相关性最强的 Top 10
print("相关性最强的行业对 (Top 10):")
print("=" * 60)
top10 = results_df.reindex(results_df['相关系数'].abs().sort_values(ascending=False).index).head(10)
print(top10[['行业A', '行业B', '相关系数', 'P值', '显著']].to_string(index=False))

# 相关性最弱的 Top 10
print("\n相关性最弱的行业对 (Top 10):")
print("=" * 60)
bottom10 = results_df.reindex(results_df['相关系数'].abs().sort_values().index).head(10)
print(bottom10[['行业A', '行业B', '相关系数', 'P值', '显著']].to_string(index=False))

In [ ]:
# 相关性分布统计

print("相关系数分布统计")
print("=" * 55)
print(f"总行业对数: {len(results_df)}")
print(f"显著相关的对数 (P<0.05): {(results_df['P值'] < 0.05).sum()}")
print(f"不显著的对数: {(results_df['P值'] >= 0.05).sum()}")

print(f"\n相关系数分布:")
print(f"  |r| > 0.7 (强相关): {(results_df['相关系数'].abs() > 0.7).sum()} 对")
print(f"  0.5 < |r| < 0.7 (中等相关): {((results_df['相关系数'].abs() > 0.5) & (results_df['相关系数'].abs() <= 0.7)).sum()} 对")
print(f"  0.3 < |r| < 0.5 (弱相关): {((results_df['相关系数'].abs() > 0.3) & (results_df['相关系数'].abs() <= 0.5)).sum()} 对")
print(f"  |r| < 0.3 (极弱/无相关): {(results_df['相关系数'].abs() <= 0.3).sum()} 对")

# 相关系数分布直方图
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(results_df['相关系数'], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Pearson 相关系数')
ax.set_ylabel('频数')
ax.set_title('行业间相关系数分布')
plt.tight_layout()
plt.show()

## 10.8 小结

| 技能 | ✓ |
|------|---|
| 理解 P 值的含义（不是 H₀ 为真的概率） | ☐ |
| 能做 JB / KS 正态性检验并解读结果 | ☐ |
| 理解检验的前提假设（正态性、方差齐性） | ☐ |
| 能做单样本 / 双样本 / 配对 t 检验 | ☐ |
| 能计算 Pearson / Spearman 相关系数 | ☐ |
| 能判断相关性是否显著（P < α） | ☐ |
| 能做 OLS 回归并解读系数、t 值、R² | ☐ |
| 理解 Q-Q 图的含义 | ☐ |

---

> 🎯 **学完这一讲，你应该能：**
> - 用统计检验判断数据是否服从正态分布
> - 用 t 检验比较策略收益是否有显著差异
> - 用相关系数分析行业/股票之间的关系，并判断是否显著
> - 用 OLS 回归做因子分析（为后续 CAPM / Fama-French 打基础）
> - 理解「尖峰厚尾」对风险管理的意义

**下一讲预告：** 用 Pandas + AKShare 构建多条件股票筛选器（阶段项目 1）。